In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Mitgliedsfunktionen
def niedrig(x, a, b):
    """Absteigende Zugehörigkeitsfunktion"""
    if x <= a:
        return 1.0
    elif x >= b:
        return 0.0
    else:
        return (b - x) / (b - a)

def mittel(x, a, b, c, d):
    """Trapezförmige Zugehörigkeitsfunktion"""
    if x <= a or x >= d:
        return 0.0
    elif a < x <= b:
        return (x - a) / (b - a)
    elif b < x <= c:
        return 1.0
    else:
        return (d - x) / (d - a)

def hoch(x, a, b):
    """Aufsteigende Zugehörigkeitsfunktion"""
    if x <= a:
        return 0.0
    elif x >= b:
        return 1.0
    else:
        return (x - a) / (b - a)

# Fuzzifizierung
def fuzzifiziere_preis(preis):
    """Preis: 0-100"""
    return {
        'niedrig': niedrig(preis, 0, 40),
        'mittel': mittel(preis, 20, 40, 60, 80),
        'hoch': hoch(preis, 60, 100)
    }

def fuzzifiziere_qualitaet(qualitaet):
    """Qualität: 0-100"""
    return {
        'niedrig': niedrig(qualitaet, 0, 40),
        'mittel': mittel(qualitaet, 20, 40, 60, 80),
        'hoch': hoch(qualitaet, 60, 100)
    }

# Regelbasis
def fuzzy_inferenz(preis_fuzzy, qualitaet_fuzzy):
    """
    Regeln:
    1. WENN Preis niedrig UND Qualität hoch DANN Eignung hoch
    2. WENN Preis mittel UND Qualität mittel DANN Eignung mittel
    3. WENN Preis hoch ODER Qualität niedrig DANN Eignung niedrig
    4. WENN Qualität hoch DANN Eignung hoch
    """

    # Regel 1: Niedriger Preis UND hohe Qualität → hohe Eignung
    regel1 = min(preis_fuzzy['niedrig'], qualitaet_fuzzy['hoch'])

    # Regel 2: Mittlerer Preis UND mittlere Qualität → mittlere Eignung
    regel2 = min(preis_fuzzy['mittel'], qualitaet_fuzzy['mittel'])

    # Regel 3: Hoher Preis ODER niedrige Qualität → niedrige Eignung
    regel3 = max(preis_fuzzy['hoch'], qualitaet_fuzzy['niedrig'])

    # Regel 4: Hohe Qualität → hohe Eignung
    regel4 = qualitaet_fuzzy['hoch']

    return {
        'niedrig': regel3,
        'mittel': regel2,
        'hoch': max(regel1, regel4)
    }

# Defuzzifizierung (Höhenmethode / Center of Gravity)
def defuzzifiziere(eignung_fuzzy):
    """Berechnet konkreten Eignungswert (0-100)"""
    # Stützstellen für die Ausgangsgröße
    niedrig_wert = 20
    mittel_wert = 50
    hoch_wert = 85

    # Gewichteter Durchschnitt
    zaehler = (eignung_fuzzy['niedrig'] * niedrig_wert +
               eignung_fuzzy['mittel'] * mittel_wert +
               eignung_fuzzy['hoch'] * hoch_wert)

    nenner = (eignung_fuzzy['niedrig'] +
              eignung_fuzzy['mittel'] +
              eignung_fuzzy['hoch'])

    if nenner == 0:
        return 50  # Neutraler Wert bei keiner Aktivierung

    return zaehler / nenner

# Hauptfunktion zur Lieferantenbewertung
def bewerte_lieferant(preis, qualitaet):
    """Bewertet einen Lieferanten"""
    preis_fuzzy = fuzzifiziere_preis(preis)
    qualitaet_fuzzy = fuzzifiziere_qualitaet(qualitaet)
    eignung_fuzzy = fuzzy_inferenz(preis_fuzzy, qualitaet_fuzzy)
    eignung = defuzzifiziere(eignung_fuzzy)

    return eignung, preis_fuzzy, qualitaet_fuzzy, eignung_fuzzy

# Test mit drei Lieferantenprofilen
lieferanten = [
    {"name": "Lieferant A", "preis": 30, "qualitaet": 85},
    {"name": "Lieferant B", "preis": 70, "qualitaet": 50},
    {"name": "Lieferant C", "preis": 45, "qualitaet": 40}
]

print("=== Fuzzy-Logik Lieferantenbewertung ===\n")

for lieferant in lieferanten:
    eignung, p_fuzz, q_fuzz, e_fuzz = bewerte_lieferant(
        lieferant["preis"],
        lieferant["qualitaet"]
    )

    print(f"{lieferant['name']}:")
    print(f"  Preis: {lieferant['preis']}, Qualität: {lieferant['qualitaet']}")
    print(f"  Eignung: {eignung:.2f}%")
    print(f"  Fuzzy-Werte Eignung: niedrig={e_fuzz['niedrig']:.2f}, "
          f"mittel={e_fuzz['mittel']:.2f}, hoch={e_fuzz['hoch']:.2f}")
    print()

=== Fuzzy-Logik Lieferantenbewertung ===

Lieferant A:
  Preis: 30, Qualität: 85
  Eignung: 85.00%
  Fuzzy-Werte Eignung: niedrig=0.00, mittel=0.00, hoch=0.62

Lieferant B:
  Preis: 70, Qualität: 50
  Eignung: 32.00%
  Fuzzy-Werte Eignung: niedrig=0.25, mittel=0.17, hoch=0.00

Lieferant C:
  Preis: 45, Qualität: 40
  Eignung: 50.00%
  Fuzzy-Werte Eignung: niedrig=0.00, mittel=1.00, hoch=0.00

